In [1]:
import pandas as pd

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from groq import Groq

import sentence_transformers
import huggingface_hub
import transformers
import langchain_huggingface



c:\Users\Yashasvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Yashasvi\AppData\Local\Temp\ipykernel_3172\3829407677.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import pandas as pd

df = pd.read_csv(
    "university_rag_dataset_10000.csv"
)

df.head()
print(df.shape)

(10000, 3)


In [3]:
print(df.columns)

Index(['category', 'question', 'answer'], dtype='str')


In [4]:
df = df.dropna()

df["category"] = df["category"].astype(str)
df["question"] = df["question"].astype(str)
df["answer"] = df["answer"].astype(str)

print(df.shape)

(10000, 3)


In [5]:
documents = []

for _, row in df.iterrows():

    content = f"""
Category: {row['category']}

Question:
{row['question']}

Answer:
{row['answer']}
"""

    doc = Document(
        page_content=content,
        metadata={
            "category": row["category"],
            "question": row["question"]
        }
    )

    documents.append(doc)

In [6]:
print(documents[0])

page_content='
Category: Fees

Question:
What is the annual tuition fee for B.Tech? (MBA)

Answer:
The annual tuition fee for B.Tech is Rs. 95,000 per academic year. This information applies to MBA unless otherwise notified by the university.
' metadata={'category': 'Fees', 'question': 'What is the annual tuition fee for B.Tech? (MBA)'}


In [7]:
#Splitting documents

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)
print(docs[0].metadata)

{'category': 'Fees', 'question': 'What is the annual tuition fee for B.Tech? (MBA)'}


In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4170.78it/s]


In [9]:
# creating  FAISS vectorstore

vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

In [10]:
# Save Vector database

vectorstore.save_local(
    "vector_db"
)

In [11]:
vectorstore = FAISS.load_local(
    "vector_db",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [12]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":20
    }
)

In [13]:
retrieved_docs = retriever.invoke(
    "What is the annual tuition fee?"
)

len(retrieved_docs)

print(retrieved_docs[0].page_content)

print(retrieved_docs[0].metadata)

Category: Fees

Question:
What is the annual tuition fee for B.Tech? (BBA)

Answer:
The annual tuition fee for B.Tech is Rs. 95,000 per academic year. This information applies to BBA unless otherwise notified by the university.
{'category': 'Fees', 'question': 'What is the annual tuition fee for B.Tech? (BBA)'}


In [14]:
sources = []

for doc in retrieved_docs:

    sources.append(
        doc.metadata.get(
            "category",
            "Unknown"
        )
    )

print(list(set(sources)))

['Fees']


In [15]:
client = Groq(
    api_key="GROQ_API_KEY"
)

In [16]:
SYSTEM_PROMPT = """
You are a University Query Management Assistant.

Rules:

1. Answer ONLY from retrieved context.

2. Never invent facts.

3. If answer not found say:

Information not available in university records.

4. Use conversation history when relevant.

5. Keep answers concise.

6. Use bullet points for rules.

7. Mention fees exactly as provided.

8. Mention examination regulations exactly.

9. Never answer outside university domain.

10. If context contains multiple answers,
choose the most relevant one.
"""

In [17]:
def build_context(retrieved_docs):

    context = "\n\n".join(
        [
            doc.page_content
            for doc in retrieved_docs
        ]
    )

    return context

In [18]:
def classify_query(query):

    return "general"

In [19]:
def validate_context(context):
    if context:
        return True

    return False

In [20]:
#It is the next version of the rag which is being used to store the chat history in memory

def get_chat_history():

    if len(chat_history) == 0:
        return "No previous conversation."

    history_text = ""

    for chat in chat_history[-MAX_HISTORY:]:

        history_text += f"""
User: {chat['question']}

Assistant: {chat['answer']}

"""

    return history_text

In [21]:
def extract_sources(docs):
    return ["university_rag_dataset_10000.csv"]
chat_history = []
MAX_HISTORY = 5

def ask_university_bot(query):

    query_type = classify_query(query)

    retrieved_docs = retriever.invoke(query)

    if not validate_context(retrieved_docs):

        return {

            "answer":
            "Information not available in university records.",

            "sources":[]
        }

    context = build_context(
        retrieved_docs
    )

    history = get_chat_history()

    user_prompt = f"""

Conversation History:

{history}

Query Category:

{query_type}

Retrieved Context:

{context}

Question:

{query}

Answer:
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        temperature=0.2,

        messages=[

            {
                "role":"system",
                "content":SYSTEM_PROMPT
            },

            {
                "role":"user",
                "content":user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return {

        "answer": answer,

        "sources":
        extract_sources(
            retrieved_docs
        )
    }

In [22]:
import os

print(os.getenv("GROQ_API_KEY"))

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


In [23]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

In [24]:
result = ask_university_bot(
    "What is the annual tuition fee?"
)

print(result["answer"])

print("\nSources:")
print(result["sources"])

The annual tuition fee for B.Tech is Rs. 95,000 per academic year. This information applies to BBA unless otherwise notified by the university.

Sources:
['university_rag_dataset_10000.csv']


In [25]:
chat_history = []
MAX_HISTORY = 5
def chat(query):

    result = ask_university_bot(query)

    chat_history.append(
        {
            "question":query,
            "answer":result["answer"]
        }
    )

    return result

In [26]:
query = "What is the annual tuition fee?"

result = chat(query)

print(result["answer"])

The annual tuition fee for B.Tech is Rs. 95,000 per academic year. This information applies to BBA unless otherwise notified by the university.


In [27]:
#It is the next version of the rag which is being used to store the chat history in memory

def get_chat_history():

    if len(chat_history) == 0:
        return "No previous conversation."

    history_text = ""

    for chat in chat_history[-MAX_HISTORY:]:

        history_text += f"""
User: {chat['question']}

Assistant: {chat['answer']}

"""

    return history_text

In [28]:
# Testing chat history retrieval

print(get_chat_history())


User: What is the annual tuition fee?

Assistant: The annual tuition fee for B.Tech is Rs. 95,000 per academic year. This information applies to BBA unless otherwise notified by the university.




In [29]:
def classify_query(query):

    query = query.lower()

    if any(word in query for word in
           ["fee","tuition","payment","cost"]):

        return "Fees"

    elif any(word in query for word in
             ["hostel","mess","room"]):

        return "Hostel"

    elif any(word in query for word in
             ["placement","package","job"]):

        return "Placement"

    elif any(word in query for word in
             ["scholarship","financial aid"]):

        return "Scholarship"

    elif any(word in query for word in
             ["exam","cgpa","semester","result"]):

        return "Examination"

    else:

        return "General"

In [30]:
print(classify_query(
    "What is hostel fee?"
))

Fees


In [31]:
def build_context(retrieved_docs):

    context_parts = []

    for i, doc in enumerate(retrieved_docs,1):

        context_parts.append(
            f"""
Document {i}

{doc.page_content}
"""
        )

    return "\n".join(context_parts)

In [32]:
def validate_context(retrieved_docs):

    if len(retrieved_docs) == 0:

        return False

    return True

In [33]:
def extract_sources(retrieved_docs):

    sources = []

    for doc in retrieved_docs:

        sources.append({

            "category":
            doc.metadata.get(
                "category",
                "Unknown"
            ),

            "question":
            doc.metadata.get(
                "question",
                "Unknown"
            )
        })

    return sources

In [1]:
def get_response(query):
    return chat(query)